### Setup

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, util
from sae_lens import (
    SAE,
    HookedSAETransformer,
)
from transformer_lens.hook_points import HookPoint
import fasttext
from functools import partial
import pickle
from jaxtyping import Float
from datasets import load_dataset
import pandas as pd
from typing import List
from rich.table import Table
from huggingface_hub import notebook_login
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Define constants
LANGUAGES = ["informal", "polite"]  # Changed from ["original", "modern"]
LAYERS = list(range(26))  # Keep as is for gemma-2b
SENTENCE_IDXS = list(range(10, 1010, 10))  # Keep as is
NUM_FEATURES = 16384

# Set plotting style
plt.style.use('ggplot')
sns.set(style="whitegrid", font_scale=1.2)
colors = plt.cm.tab10.colors

# Create directory for saving results
os.makedirs("results", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/data", exist_ok=True)

# Function to load SAE activations
def load_activations(layers=0, base_path="./sae_acts_2b/sentence_corrector"):  # Changed path
    """
    Load SAE activations from files

    Args:
        base_path: Path to the directory containing activation files

    Returns:
        Dict of activations by language, layer, and sentence index
    """
    print("Loading SAE activations...")
    activations = {}

    for lang in LANGUAGES:
        activations[lang] = {}
        if type(layers) is int:
            layers = [layers]
        for layer in layers:
            all_sentences = []
            max_tokens = 0

            # First pass: collect activations and determine max_tokens
            for sent_idx in SENTENCE_IDXS:
                # File naming matches your generation: informal_layer{layer}_idx{sent_idx}.pt
                file_path = f"{base_path}/{lang}_layer{layer}_idx{sent_idx}.pt"
                try:
                    act = torch.load(file_path)
                    num_tokens = act.shape[1]
                    max_tokens = max(max_tokens, num_tokens)
                    all_sentences.append(act)

                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
                    all_sentences.append(None)

            # Second pass: pad all activations to max_tokens
            padded_sentences = []
            for act in all_sentences:
                if act is not None:
                    num_tokens = act.shape[1]
                    if num_tokens < max_tokens:
                        padding = (0, 0, 0, max_tokens - num_tokens)
                        act = torch.nn.functional.pad(act, padding)
                    padded_sentences.append(act)

            # Concatenate all padded activations along the first dimension
            activations[lang][layer] = torch.cat(padded_sentences, dim=0)

    print("Finished loading activations")
    return activations

In [ ]:
import tqdm
tqdm.tqdm = lambda *args, **kwargs: iter(args[0])
torch.set_grad_enabled(False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
notebook_login()

In [ ]:
# CHANGE: Updated to gemma-2-2b to match your model
gemma_2b: HookedSAETransformer = HookedSAETransformer.from_pretrained("gemma-2-2b", device=device)

You tried to specify center_unembed=True for a model using logit softcap, but this can't be done! Softcapping is not invariant upon adding a constant Setting center_unembed=False instead.


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

You are not using LayerNorm, so the writing weights can't be centered! Skipping


Loaded pretrained model gemma-2-9b-it into HookedTransformer


### Steering Helper Functions

In [ ]:
GENERATE_KWARGS = dict(temperature=0.5, freq_penalty=1.0, verbose=False)

def steering_hook(
    activations: Float[torch.Tensor, "batch pos d_in"],
    hook: HookPoint,
    sae: SAE,
    latent_idx: int,
    steering_coefficient: float,
) -> torch.Tensor:
    """
    Steers the model by returning a modified activations tensor, with some multiple of the steering vector added to all
    sequence positions.
    """
    return activations + steering_coefficient * sae.W_dec[latent_idx]

def generate_with_multi_steering(
    model: HookedSAETransformer,
    steer_saes: list[SAE],
    latent_idxs: List[int],
    steering_coefficients: List[float],
    monitor_saes: list[SAE],
    prompts: List[str],
    max_new_tokens: int = 50,
):
    """
    Generates text with multiple steering vectors applied from different SAEs.
    Each SAE modifies activations at its own hook point with its own latent and coefficient.
    """
    assert len(steer_saes) == len(latent_idxs) == len(steering_coefficients), "Mismatch in lengths of SAE-related lists."

    # Create individual hook functions for each SAE
    hooks = []
    for sae, latent_idx, coeff in zip(steer_saes, latent_idxs, steering_coefficients):
        hook_fn = partial(
            steering_hook,
            sae=sae,
            latent_idx=latent_idx,
            steering_coefficient=coeff,
        )
        # FIXED: Use cfg.hook_name or metadata.get('hook_name')
        hook_name = sae.cfg.hook_name if hasattr(sae.cfg, 'hook_name') else sae.cfg.metadata.get('hook_name')
        hooks.append((hook_name, hook_fn))

    with model.hooks(fwd_hooks=hooks):
        if monitor_saes:
            _, cache = model.run_with_cache_with_saes(prompts, saes=monitor_saes)
        output = model.generate(prompts, max_new_tokens=max_new_tokens, **GENERATE_KWARGS)
    if monitor_saes:
        # FIXED: Same fix for monitor SAEs
        monitor_activations = []
        for sae in monitor_saes:
            hook_name = sae.cfg.hook_name if hasattr(sae.cfg, 'hook_name') else sae.cfg.metadata.get('hook_name')
            monitor_activations.append(cache[f"{hook_name}.hook_sae_acts_post"])
        del cache
        return output, monitor_activations
    return output

def generate_with_multi_steering_batch(
    model,
    steer_layers: List[int],
    steer_feature_idx: List[int],
    steer_feature_diff: List[float],
    monitor_layers: List[int] = [],
    prompts: List[str] = ['I was learning'],
):
    assert len(steer_layers) == len(steer_feature_idx) == len(steer_feature_diff), "All input lists must be the same length."

    # CHANGE: Updated release to gemma-scope-2b-pt-res-canonical
    steer_saes = []
    for layer in steer_layers:
        sae = SAE.from_pretrained(
            release='gemma-scope-2b-pt-res-canonical',  # CHANGE: Updated for 2b model
            sae_id=f"layer_{layer}/width_16k/canonical",
            device=str(device),
        )[0]  # Get just the SAE object
        steer_saes.append(sae)
    monitor_saes = []
    for layer in monitor_layers:
        sae = SAE.from_pretrained(
            release='gemma-scope-2b-pt-res-canonical',  # CHANGE: Updated for 2b model
            sae_id=f"layer_{layer}/width_16k/canonical",
            device=str(device),
        )[0]  # Get just the SAE object
        monitor_saes.append(sae)

    generated_sentences = {}

    # No steering generation
    generated_sentences[0] = model.generate(prompts, max_new_tokens=50, **GENERATE_KWARGS)

    # Multi-steered generation
    if monitor_layers:
        steered_output, monitor_activations = generate_with_multi_steering(
            model=model,
            steer_saes=steer_saes,
            latent_idxs=steer_feature_idx,
            steering_coefficients=[diff for diff in steer_feature_diff],
            monitor_saes = monitor_saes,
            prompts=prompts,
            max_new_tokens=50,
        )
        generated_sentences[1] = steered_output
        return generated_sentences, monitor_activations
    else:
        steered_output = generate_with_multi_steering(
            model=model,
            steer_saes=steer_saes,
            latent_idxs=steer_feature_idx,
            steering_coefficients=[diff for diff in steer_feature_diff],
            monitor_saes = monitor_saes,
            prompts=prompts,
            max_new_tokens=50,
        )
        generated_sentences[1] = steered_output
        return generated_sentences

def generate_with_steering_batch(
    model,
    steer_layer: int,
    steer_feature_idxs: List[int],
    steer_feature_diffs: List[float],
    prompts: List[str] = ['I was learning'],
):
    # CHANGE: Updated release to gemma-scope-2b-pt-res-canonical
    sae = SAE.from_pretrained(
            release='gemma-scope-2b-pt-res-canonical',  # CHANGE: Updated for 2b model
            sae_id=f"layer_{steer_layer}/width_16k/canonical",
            device=str(device),
        )[0]  # Get just the SAE object
    generated_sentences = {}
    generated_sentences[0] = model.generate(prompts, max_new_tokens=50, **GENERATE_KWARGS)
    generated_sentences[-1] = model.generate(prompts, max_new_tokens=50, **GENERATE_KWARGS)
    for i in range(len(steer_feature_idxs)):
        steer_feature_idx = [steer_feature_idxs[i]]
        steer_feature_diff = [steer_feature_diffs[i]]
        steered_output = generate_with_multi_steering(
            model=model,
            steer_saes=[sae],
            latent_idxs=steer_feature_idx,
            steering_coefficients=[diff for diff in steer_feature_diff],
            monitor_saes = [],
            prompts=prompts,
            max_new_tokens=50,
        )
        generated_sentences[i+1] = steered_output
    del sae
    return generated_sentences

### Feature-filtering Helper Functions

Finding Top difference features

In [ ]:
def diff_layer(sentence_activations, layer, diff_type="abs", k=10, activation_type="mean"):
    # Changed keys to match informal/polite
    informal_act = sentence_activations['informal'][layer]
    polite_act = sentence_activations['polite'][layer]

    top_features = np.array([])

    if diff_type == "abs":
        # polite - informal: positive values = more polite features
        feature_diff = torch.mean(polite_act, axis=0) - torch.mean(informal_act, axis=0)
        valid_indices = feature_diff > 0  # Keep only features where polite > informal
        sorted_indices = np.argsort(feature_diff[valid_indices])[-k:]
        top_features = valid_indices.nonzero().squeeze()[sorted_indices]
        top_diffs = feature_diff[top_features]

    return top_features, top_diffs

### Scoring Features in Every Layer

In [ ]:
# Load the Dolly dataset
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
df = pd.DataFrame(dataset)

# Drop empty instructions
df = df[df["instruction"].str.strip().astype(bool)]
df = df[df["instruction"].str.len() <= 100]

# Filter to only 'general_qa' and 'open_qa' categories
df = df[df["category"].isin(["general_qa", "open_qa"])]

# Sample 100 prompts
sampled_prompts = df["instruction"].sample(n=100, random_state=42).reset_index(drop=True)

# Convert to list
prompt_list = sampled_prompts.tolist()

# Print first 5
for prompt in prompt_list[-5:]:
    print(prompt)


In [ ]:
# Create output directory for informal/polite results
os.makedirs("gemma2b_informal_polite_similarity", exist_ok=True)

for layer in LAYERS:
    # Load activations with updated path
    activations = load_activations(layer, base_path="./sae_acts_2b/sentence_corrector")

    # Load token length files with your naming convention
    with open("./token_lengths/polite_informal.pkl", "rb") as f:
        informal_token_length = pickle.load(f)
    with open("./token_lengths/polite_polite.pkl", "rb") as f:
        polite_token_length = pickle.load(f)

    token_length = {}
    token_length["informal"] = torch.tensor(informal_token_length)[:1000]
    token_length["polite"] = torch.tensor(polite_token_length)[:1000]

    sentence_mean_activations = {}
    sentence_last_activations = {}
    for lang in LANGUAGES:
        activations[lang][layer] = activations[lang][layer][:1000]
        sentence_mean_activations[lang] = {}
        sentence_last_activations[lang] = {}
        indices = torch.arange(activations[lang][layer].shape[1]).unsqueeze(0).expand(1000, -1)
        mask = indices < token_length[lang].unsqueeze(1)
        mask_expanded = mask.unsqueeze(-1)
        masked_tensor = activations[lang][layer][:1000] * mask_expanded
        last_token_indices = token_length[lang] - 1
        sum_tensor = masked_tensor.sum(dim=1)
        sentence_mean_activations[lang][layer] = sum_tensor / token_length[lang].unsqueeze(1)
        sentence_last_activations[lang][layer] = masked_tensor[torch.arange(1000), last_token_indices]
    del activations, token_length

    print(f"LAYER:{layer}")
    feature_index, feature_diff = diff_layer(sentence_mean_activations, layer, diff_type="abs", k=3, activation_type="mean")
    print(feature_index)
    print(feature_diff)

    generated_sentences = generate_with_steering_batch(gemma_2b, layer, feature_index, feature_diff, prompts=prompt_list)
    model_score_pairs = list(zip(feature_index, feature_diff, generated_sentences))

    # Save with updated path
    with open(f"gemma2b_informal_polite_similarity/similarity{layer}_mean.pkl", "wb") as f:
        pickle.dump(model_score_pairs, f)

    feature_index, feature_diff = diff_layer(sentence_last_activations, layer, diff_type="abs", k=3, activation_type="last")
    print(feature_index)
    print(feature_diff)

    generated_sentences = generate_with_steering_batch(gemma_2b, layer, feature_index, feature_diff, prompts=prompt_list)
    model_score_pairs = list(zip(feature_index, feature_diff, generated_sentences))

    with open(f"gemma2b_informal_polite_similarity/similarity{layer}_last.pkl", "wb") as f:
        pickle.dump(model_score_pairs, f)